# FlagOS Track 1 — `log10` (university entry)

**Full repo (panel review):** https://github.com/viniciuserrav/flagos-track1
**License:** Apache-2.0

This notebook contains the leaderboard-scored op (`log10`) end-to-end:
1. Triton kernel + `torch.log10` fallback for CPU runs,
2. Correctness vs `torch.log10` across fp16/bf16/fp32 and edge values,
3. Microbenchmark vs `torch.log10` (only meaningful with a GPU),
4. `submission.csv` generator for the Kaggle leaderboard.

Additional Track-1 operators land in the GitHub repo as they are validated. Each is marked **completed** only with kernel + tests + benchmark + notebook integration.

## 1. Environment

In [ ]:
import math, os
import numpy as np
import pandas as pd
import torch
print('torch :', torch.__version__, '| cuda:', torch.cuda.is_available())
try:
    import triton, triton.language as tl
    HAS_TRITON = True
    print('triton:', triton.__version__)
except Exception as e:
    HAS_TRITON = False
    print('triton: not available ->', e)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_TRITON = HAS_TRITON and DEVICE == 'cuda'
if DEVICE == 'cuda':
    print('gpu   :', torch.cuda.get_device_name(0))


## 2. Triton `log10` kernel + `torch.log10` fallback

Identity: `log10(x) = ln(x) * 0.4342944819032518`. fp16/bf16 are promoted to fp32 inside the kernel and cast back on store (matches PyTorch). fp32 stays in fp32. fp64 falls back to `torch.log10` (Triton lacks fp64 transcendentals on T4-class HW).

In [ ]:
RECIP_LN10 = 1.0 / math.log(10.0)

# Probe CUDA capability: Kaggle sometimes hands out a Tesla P100 (sm_60) which the
# current torch wheel does not support; fall back to CPU in that case.
if DEVICE == 'cuda':
    try:
        _probe = torch.zeros(1, device='cuda')
        cap = torch.cuda.get_device_capability(0)
        if cap[0] < 7:
            print(f'gpu sm_{cap[0]}{cap[1]} not supported by torch wheel; falling back to CPU')
            DEVICE = 'cpu'
            USE_TRITON = False
    except Exception as e:
        print('cuda probe failed -> falling back to CPU:', e)
        DEVICE = 'cpu'
        USE_TRITON = False
if HAS_TRITON:
    @triton.autotune(
        configs=[triton.Config({'BLOCK_SIZE': bs}, num_warps=nw, num_stages=ns)
                 for bs in (1024, 2048, 4096, 8192) for nw in (4, 8) for ns in (2, 3)],
        key=['n_elements'],
    )
    @triton.jit
    def _log10_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
        pid = tl.program_id(0)
        offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offs < n_elements
        x = tl.load(x_ptr + offs, mask=mask, other=1.0)
        y = (tl.log(x.to(tl.float32)) * 0.4342944819032518).to(x.dtype)
        tl.store(y_ptr + offs, y, mask=mask)

def log10(x: torch.Tensor, *, out=None) -> torch.Tensor:
    if not x.is_floating_point():
        x = x.to(torch.float32)
    if not (USE_TRITON and x.is_cuda and x.dtype in (torch.float16, torch.bfloat16, torch.float32)):
        return torch.log10(x, out=out) if out is not None else torch.log10(x)
    if not x.is_contiguous():
        x = x.contiguous()
    if out is None:
        out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    grid = lambda meta: (triton.cdiv(n, meta['BLOCK_SIZE']),)
    _log10_kernel[grid](x, out, n)
    return out

def log10_(x: torch.Tensor) -> torch.Tensor:
    return log10(x, out=x)


## 3. Correctness vs `torch.log10`

In [ ]:
TOL = {torch.float16: (1e-3, 1e-3), torch.bfloat16: (1e-2, 1.6e-2), torch.float32: (1e-5, 1.3e-6)}
for dtype, (rtol, atol) in TOL.items():
    torch.manual_seed(0)
    x = torch.rand(1024, 1024, device=DEVICE, dtype=dtype) + 0.1
    torch.testing.assert_close(log10(x), torch.log10(x), equal_nan=True, rtol=rtol, atol=atol)
    print(f'ok  random   {str(dtype):>16}')
edge = torch.tensor([0., -1., 1., 10., 1e-30, 1e30, float('inf'), -float('inf'), float('nan')],
                    device=DEVICE, dtype=torch.float32)
torch.testing.assert_close(log10(edge), torch.log10(edge), equal_nan=True, rtol=1e-5, atol=1.3e-6)
print('ok  edge values:', [round(v, 4) for v in log10(edge).cpu().tolist()])
for shape in [(1,), (33,), (3, 17, 5), (128, 256), (1024, 1024)]:
    x = torch.rand(shape, device=DEVICE, dtype=torch.float32) + 0.1
    torch.testing.assert_close(log10(x), torch.log10(x), rtol=1e-5, atol=1.3e-6)
print('ok  multi-shape')


## 4. Microbenchmark vs `torch.log10` (CUDA only)

In [ ]:
if DEVICE == 'cuda' and HAS_TRITON:
    import triton.testing as tt
    sizes = [1 << k for k in (10, 13, 16, 18, 20, 22, 24, 26)]
    rows = []
    for n in sizes:
        x = torch.rand(n, device='cuda', dtype=torch.float32) + 0.1
        log10(x); torch.log10(x); torch.cuda.synchronize()
        tri_ms = tt.do_bench(lambda: log10(x), warmup=25, rep=100)
        ref_ms = tt.do_bench(lambda: torch.log10(x), warmup=25, rep=100)
        rows.append({
            'elements': n,
            'triton_ms': tri_ms,
            'torch_ms': ref_ms,
            'speedup': ref_ms / tri_ms,
            'bandwidth_gbs': (2 * n * 4) / (tri_ms * 1e-3) / 1e9,
        })
    bench = pd.DataFrame(rows)
    print(bench.to_string(index=False))
    print(f'\ngeomean speedup vs torch.log10: {float(np.exp(np.log(bench.speedup).mean())):.3f}x')
    print(f'peak effective bandwidth      : {float(bench.bandwidth_gbs.max()):.0f} GB/s')
else:
    print('CUDA / Triton unavailable, skipping benchmark.')


## 5. `submission.csv`

In [ ]:
NUM_ROWS = 1000
x = np.linspace(0.001, 1000.0, NUM_ROWS, dtype=np.float64)
target = np.log10(x)
submission = pd.DataFrame({'ID': np.arange(NUM_ROWS, dtype=np.int64), 'target': target})
submission.to_csv('submission.csv', index=False)
print(f'wrote submission.csv rows={len(submission)}')
print(submission.head(3))
print('...')
print(submission.tail(3))


## Full Track-1 work

Repo: https://github.com/viniciuserrav/flagos-track1

Currently in the repo: log10 kernel, 16-case pytest suite, microbenchmark script, design notes in `docs/OPERATORS.md`. Additional operators land in subsequent commits, each marked **completed** vs **experimental** in the README status table.